In [ ]:
# Parameters
city_key = "ywg"
analysis_feed_id = "current"
save_figures = False
figures_dir = "reports/pr2/figures"
dpi = 200

# PR2: Temporal Route Clustering

This notebook clusters routes by their hourly departure profiles to identify service-pattern groups. The output is descriptive and supports interpretation, not a formal policy ranking.

## 1. Setup & Helpers

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

from ptn_analysis.analysis import WEB_MERCATOR, add_consistent_basemap
from ptn_analysis.context.reporting import (
    ensure_report_dirs,
    save_placeholder_figure,
    save_report_figure,
)

ensure_report_dirs("pr2")
figure_output_directory = Path(figures_dir)
figure_output_directory.mkdir(parents=True, exist_ok=True)

from ptn_analysis import TransitContext
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## 2. Load Route-Hour Profiles

In [ ]:
ctx = TransitContext.from_defaults(feed_id=analysis_feed_id)
frequency_analyzer = ctx.frequency()
hourly_departure_table = frequency_analyzer.build_hourly_departure_table()

print(f"Hourly rows: {len(hourly_departure_table)}")
display(hourly_departure_table.head())

## 3. Elbow Curve

In [ ]:

if hourly_departure_table.empty:
    hourly_profile_table = pd.DataFrame()
    feature_matrix = None
    save_placeholder_figure("clustering_elbow.png", "Hourly departure data is missing.", figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
else:
    hourly_profile_table = hourly_departure_table.pivot_table(
        index="route_short_name",
        columns="hour",
        values="departures",
        fill_value=0,
    )
    row_totals = hourly_profile_table.sum(axis=1)
    hourly_profile_table = hourly_profile_table.div(row_totals.replace(0, 1), axis=0)

    scaler = StandardScaler()
    feature_matrix = scaler.fit_transform(hourly_profile_table)
    candidate_k_values = [2, 3, 4, 5, 6, 7, 8]
    inertia_values = []
    silhouette_values = []
    for cluster_count in candidate_k_values:
        model = KMeans(n_clusters=cluster_count, random_state=42, n_init=10)
        cluster_labels = model.fit_predict(feature_matrix)
        inertia_values.append(model.inertia_)
        silhouette_values.append(silhouette_score(feature_matrix, cluster_labels))

    figure, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(candidate_k_values, inertia_values, marker="o")
    axes[0].set_title("Elbow Curve")
    axes[0].set_xlabel("Cluster count")
    axes[0].set_ylabel("Inertia")
    axes[1].plot(candidate_k_values, silhouette_values, marker="o", color="#1b9e77")
    axes[1].set_title("Silhouette Score")
    axes[1].set_xlabel("Cluster count")
    axes[1].set_ylabel("Score")
    save_report_figure(figure, "clustering_elbow.png", figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
    plt.close(figure)


In [ ]:
# === FINAL REPORT ADDITION: Calinski-Harabasz, Davies-Bouldin, DBSCAN comparison ===
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from sklearn.cluster import DBSCAN

if feature_matrix is not None:
    # Full validation metrics for k=2..8
    validation_metrics = []
    for k in candidate_k_values:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(feature_matrix)
        validation_metrics.append({
            "k": k,
            "silhouette": round(silhouette_score(feature_matrix, labels), 3),
            "calinski_harabasz": round(calinski_harabasz_score(feature_matrix, labels), 1),
            "davies_bouldin": round(davies_bouldin_score(feature_matrix, labels), 3),
            "inertia": round(km.inertia_, 1),
        })
    validation_df = pd.DataFrame(validation_metrics)
    print("=== Clustering Validation Metrics ===")
    print(validation_df.to_string(index=False))

    # DBSCAN comparison
    dbscan = DBSCAN(eps=1.5, min_samples=3)
    dbscan_labels = dbscan.fit_predict(feature_matrix)
    n_dbscan_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
    n_noise = list(dbscan_labels).count(-1)
    print(f"\nDBSCAN: {n_dbscan_clusters} clusters, {n_noise} noise points ({len(feature_matrix)} total)")
    if n_dbscan_clusters >= 2:
        mask = dbscan_labels != -1
        db_sil = silhouette_score(feature_matrix[mask], dbscan_labels[mask])
        print(f"DBSCAN silhouette (excl noise): {db_sil:.3f}")
        print(f"K-means (k=4) silhouette: {silhouette_values[2]:.3f}")
else:
    print("No feature matrix available for validation")

## 4. Final Route Clusters

In [ ]:
# Fit final k-means on route hourly departure profiles
from joblib import dump
from ptn_analysis.analysis.frequency import MODELS_DIR

if not hourly_departure_table.empty:
    final_cluster_count = 4
    final_model = KMeans(n_clusters=final_cluster_count, random_state=42, n_init=10)
    final_labels = final_model.fit_predict(feature_matrix)
    hourly_profile_table = hourly_profile_table.copy()
    hourly_profile_table["cluster"] = final_labels
    cluster_profile_table = hourly_profile_table.groupby("cluster").mean().T
    cluster_route_table = hourly_profile_table[["cluster"]].reset_index()

    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    dump(final_model, MODELS_DIR / "route_clusters_v1.pkl")
    print(f"Route cluster model saved -> {MODELS_DIR / 'route_clusters_v1.pkl'}")
    display(cluster_route_table.head())
else:
    cluster_profile_table = None
    cluster_route_table = None

## 5. Neighbourhood Service Clustering

Cluster neighbourhoods by transit service features to identify service typologies across Winnipeg.

In [ ]:
import geopandas as gpd
from joblib import dump
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from ptn_analysis.analysis.frequency import MODELS_DIR

ca = ctx.coverage()
density_table = ca.neighbourhood_density()
jobs_table = ca.jobs_access()

nb_features = density_table.merge(
    jobs_table[["neighbourhood", "jobs_access_score"]],
    on="neighbourhood", how="left"
).fillna(0)

has_temporal = cluster_profile_table is not None
has_spatial = not nb_features.empty and len(nb_features) >= 4

# Shared 4-color palette across both panels
CLUSTER_PALETTE = {0: "#1f77b4", 1: "#ff7f0e", 2: "#2ca02c", 3: "#d62728"}

if not has_temporal and not has_spatial:
    save_placeholder_figure(
        "clustering_combined.png", "Clustering data not available.",
        figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures,
    )
else:
    # Vertically stacked: temporal on top, spatial on bottom
    fig, axes = plt.subplots(2, 1, figsize=(8, 10),
                             gridspec_kw={"height_ratios": [1, 1.3]})

    # TOP: Route temporal clusters
    ax_top = axes[0]
    if has_temporal:
        cluster_names = {0: "Flat all-day", 1: "Peak-only", 2: "Extended-evening", 3: "Low-frequency"}
        for cid in cluster_profile_table.columns:
            label = cluster_names.get(cid, f"Cluster {cid}")
            n_routes = (cluster_route_table["cluster"] == cid).sum()
            ax_top.plot(
                cluster_profile_table.index, cluster_profile_table[cid],
                label=f"{label} (n={n_routes})", linewidth=2.5,
                color=CLUSTER_PALETTE.get(cid),
            )
        ax_top.set_xlabel("Hour of day", fontsize=11)
        ax_top.set_ylabel("Share of daily departures", fontsize=11)
        ax_top.legend(fontsize=10, loc="upper right")
        ax_top.grid(alpha=0.3)
    else:
        ax_top.text(0.5, 0.5, "No temporal data", ha="center", va="center")

    # BOTTOM: Neighbourhood service clusters map
    ax_bot = axes[1]
    if has_spatial:
        feature_cols = ["stop_count", "stop_density_per_km2", "jobs_access_score"]
        X_nb = StandardScaler().fit_transform(nb_features[feature_cols])
        km_nb = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_nb)
        nb_features["service_cluster"] = km_nb.labels_

        # Serialize neighbourhood cluster model
        MODELS_DIR.mkdir(parents=True, exist_ok=True)
        dump(km_nb, MODELS_DIR / "neighbourhood_clusters_v1.pkl")
        print(f"Neighbourhood cluster model saved -> {MODELS_DIR / 'neighbourhood_clusters_v1.pkl'}")

        neigh_gdf = ctx.working_db.neighbourhood_gdf()
        map_gdf = neigh_gdf.merge(
            nb_features[["neighbourhood", "service_cluster", "stop_density_per_km2"]],
            on="neighbourhood", how="left",
        ).to_crs(WEB_MERCATOR)

        cluster_summary = nb_features.groupby("service_cluster")[feature_cols].mean().round(1)
        for c in sorted(nb_features["service_cluster"].unique()):
            row = cluster_summary.loc[c]
            n = (nb_features["service_cluster"] == c).sum()
            label = f"C{c}: {n} nb, {row['stop_density_per_km2']:.0f} stops/km\u00b2"
            subset = map_gdf[map_gdf["service_cluster"] == c]
            subset.plot(ax=ax_bot, color=CLUSTER_PALETTE.get(c, "#bdbdbd"),
                        edgecolor="gray", linewidth=0.3, alpha=0.7, label=label)
        map_gdf[map_gdf["service_cluster"].isna()].plot(
            ax=ax_bot, color="lightgray", edgecolor="gray", linewidth=0.3, alpha=0.5)

        # Neighbourhood labels
        labeled = map_gdf.dropna(subset=["service_cluster"]).copy()
        labeled["centroid_pt"] = labeled.geometry.centroid
        labeled["area_m2"] = labeled.geometry.area
        top_labels = labeled.groupby("service_cluster").apply(
            lambda g: g.nlargest(4, "area_m2"), include_groups=False
        ).reset_index(drop=True)
        dense_labels = labeled.nlargest(6, "stop_density_per_km2")
        top_labels = pd.concat([top_labels, dense_labels]).drop_duplicates(subset="neighbourhood")
        for _, row in top_labels.iterrows():
            pt = row["centroid_pt"]
            name = row["neighbourhood"]
            short = name if len(name) <= 16 else name[:14] + ".."
            ax_bot.annotate(
                short, (pt.x, pt.y), fontsize=4.5, ha="center", va="center",
                fontweight="bold", color="#222222",
                bbox=dict(boxstyle="round,pad=0.08", facecolor="white", alpha=0.8, edgecolor="none"),
            )

        add_consistent_basemap(ax_bot)
        ax_bot.legend(loc="lower left", fontsize=9)
    else:
        ax_bot.text(0.5, 0.5, "No spatial data", ha="center", va="center")

    fig.subplots_adjust(top=0.97, bottom=0.02, left=0.08, right=0.97, hspace=0.15)
    save_report_figure(fig, "clustering_combined.png",
                       figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
    plt.close(fig)

## 6. Hierarchical Clustering Comparison

Ward's linkage hierarchical clustering as a comparison to K-means. We cut at k=4
to match the K-means solution and measure agreement via the Adjusted Rand Index.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.metrics import adjusted_rand_score, silhouette_score

if has_spatial and 'X_nb' in dir() and X_nb is not None:
    # Ward linkage on the same scaled neighbourhood features
    Z = linkage(X_nb, method='ward')

    # Dendrogram
    fig_hc, ax_hc = plt.subplots(1, 1, figsize=(12, 5))
    dendrogram(
        Z, truncate_mode='lastp', p=20, leaf_rotation=90,
        leaf_font_size=8, ax=ax_hc, color_threshold=Z[-4, 2],
    )
    ax_hc.set_title("Ward Hierarchical Clustering Dendrogram (truncated to 20 leaves)")
    ax_hc.set_xlabel("Neighbourhood cluster (size)")
    ax_hc.set_ylabel("Ward distance")
    ax_hc.axhline(y=Z[-4, 2], color='red', linestyle='--', alpha=0.5, label='k=4 cut')
    ax_hc.legend()
    save_report_figure(fig_hc, "neighbourhood_clustering.png",
                       figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
    plt.close(fig_hc)

    # Cut dendrogram at k=4 and compare with K-means
    hc_labels = fcluster(Z, t=4, criterion='maxclust') - 1  # 0-indexed
    km_labels = km_nb.labels_

    ari = adjusted_rand_score(km_labels, hc_labels)
    hc_sil = silhouette_score(X_nb, hc_labels)
    km_sil = silhouette_score(X_nb, km_labels)

    print("=== Hierarchical vs K-Means (k=4) ===")
    print(f"Adjusted Rand Index:       {ari:.3f}")
    print(f"K-Means silhouette:        {km_sil:.3f}")
    print(f"Hierarchical silhouette:   {hc_sil:.3f}")
    print(f"Agreement: {'Strong' if ari > 0.7 else 'Moderate' if ari > 0.4 else 'Weak'}"
          f" (ARI={ari:.3f})")
else:
    print("Skipped: no spatial neighbourhood features available")